# NeuroShield Track 1 - Train Model A (ResNet-50 reference baseline)
Trains 3 seeds (42, 123, 2024) on the fixed OASIS split from the NeuroShield OASIS Model B dataset, saves checkpoints + test metrics. Runs from the public NeuroShield repo code.

In [1]:
# get the package (torch/torchvision/nibabel/sklearn/pandas are already in the Kaggle image)
!git clone -q https://github.com/pankurajasaivardhan/Neuroshield.git /kaggle/working/neuroshield_repo
import sys
sys.path.insert(0, '/kaggle/working/neuroshield_repo')
print('repo ready')

repo ready


In [2]:
# rewrite split CSV scan_paths to absolute Kaggle paths (CSVs carry relative ../../data/oasis/... paths)
import os, pandas as pd
import glob
ROOT = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/splits/train.csv', recursive=True)[0]))
print('dataset ROOT:', ROOT)
OUT_DATA = '/kaggle/working/track1_data'
os.makedirs(f'{OUT_DATA}/splits', exist_ok=True)
for s in ['train', 'val', 'test']:
    df = pd.read_csv(f'{ROOT}/splits/{s}.csv')
    df['scan_path'] = df['scan_path'].apply(lambda x: os.path.join(ROOT, 'data', str(x).split('../../data/oasis/')[-1]))
    df.to_csv(f'{OUT_DATA}/splits/{s}.csv', index=False)
    missing = [p for p in df['scan_path'] if not os.path.exists(p)]
    print(s, 'rows:', len(df), 'missing_files:', len(missing))
    assert not missing, missing[:3]
    assert 'label' in df.columns and 'ID' in df.columns
print(pd.read_csv(f'{OUT_DATA}/splits/test.csv')['label'].value_counts().to_dict())

dataset ROOT: /kaggle/input/datasets/mohanroyal273/neuroshield-oasis-model-b
train rows: 164 missing_files: 0
val rows: 35 missing_files: 0
test rows: 36 missing_files: 0
{0: 21, 1: 15}


In [3]:
# train Model A (ResNet-50), 3 seeds, early stop on val loss, then test eval per seed
OUT = '/kaggle/working/outputs/track1/model_a'
!PYTHONPATH=/kaggle/working/neuroshield_repo python -m track1_alzheimers.src.train_model_a --data-root /kaggle/working/track1_data --out "$OUT" --seeds 42 123 2024

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 215MB/s]
[seed 42] epoch 1: val_loss=0.6701
[seed 42] epoch 2: val_loss=0.6137
[seed 42] epoch 3: val_loss=0.6412
[seed 42] epoch 4: val_loss=0.6874
[seed 42] epoch 5: val_loss=0.7336
[seed 42] epoch 6: val_loss=0.8108
[seed 42] epoch 7: val_loss=0.9186
[seed 42] test accuracy=0.6944
[seed 123] epoch 1: val_loss=0.6492
[seed 123] epoch 2: val_loss=0.5898
[seed 123] epoch 3: val_loss=0.6137
[seed 123] epoch 4: val_loss=0.6376
[seed 123] epoch 5: val_loss=0.7518
[seed 123] epoch 6: val_loss=0.7656
[seed 123] epoch 7: val_loss=0.8092
[seed 123] test accuracy=0.7222
[seed 2024] epoch 1: val_loss=0.6533
[seed 2024] epoch 2: val_loss=0.5878
[seed 2024] epoch 3: val_loss=0.5974
[seed 2024] epoch 4: val_loss=0.6112
[seed 2024] epoch 5: val_loss=0.6567
[seed 2024] epoch 6: val_loss=0.6761
[see

In [4]:
import json
print(json.dumps(json.load(open('/kaggle/working/outputs/track1/model_a/model_a_resnet50_results_summary.json'))['summary'], indent=2))

{
  "accuracy": {
    "mean": 0.7129629629629629,
    "std": 0.013094570021973107
  },
  "balanced_accuracy": {
    "mean": 0.7063492063492062,
    "std": 0.018374344290143185
  },
  "sensitivity_recall": {
    "mean": 0.6666666666666666,
    "std": 0.05443310539518172
  },
  "specificity": {
    "mean": 0.746031746031746,
    "std": 0.022447834323382432
  },
  "f1": {
    "mean": 0.6582854406130267,
    "std": 0.02791163743694138
  },
  "roc_auc": {
    "mean": 0.7269841269841271,
    "std": 0.006857926664981872
  }
}
